# 話者分離 手法比較ノートブック

同じ音声に対して複数の話者分離アプローチを回し、横並びで比較して採用候補を決める。
（issue #22 評価品質レイヤ / 将来の Eatas 面談転用 / [調査プロンプト](../../docs/research/step2_speaker_diarization_deepresearch_prompt.md)）

## 候補
| | STT | 話者分離 | 鍵/依存 |
|---|---|---|---|
| **A. Gladia** | Gladia(置換) | 組込み | `GRADIA_API_KEY` |
| **B. sherpa-onnx** | Whisper維持 | ローカルONNX | `diar_models/`(DL済) |
| **C. VADのみ** | Whisper維持 | 無し(最長話者=応募者) | librosa |

## 使い方
1. カーネル = **`backend/.venv`**。
2. `backend/.env` に `OPENAI_API_KEY` と（A用に）`GRADIA_API_KEY`。
3. `AUDIO` を設定（未設定なら `InputData/` の先頭ファイルを自動使用）。
4. 上から実行。**理想は2話者(面接Q&A)の音声**。単一話者だと分離の優劣が出にくい。


## 0. セットアップ

In [3]:
from pathlib import Path
import sys, os, json, time, subprocess

def _find_eval_dir() -> Path:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        cand = base / "experiments" / "evaluation"
        if (cand / "transcribe.py").exists():
            return cand
        if base.name == "evaluation" and (base / "transcribe.py").exists():
            return base
    raise RuntimeError("experiments/evaluation が見つかりません")

EVAL_DIR = _find_eval_dir()
REPO = EVAL_DIR.parents[1]
if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))
from transcribe import find_fillers, load_api_key   # 既存スパイクを再利用

MODELS_DIR = EVAL_DIR / "diar_models"

def load_gladia_key():
    for name in ("GRADIA_API_KEY", "GLADIA_API_KEY"):
        if os.environ.get(name):
            return os.environ[name]
    env = REPO / "backend" / ".env"
    if env.exists():
        for line in env.read_text().splitlines():
            line = line.strip()
            for name in ("GRADIA_API_KEY", "GLADIA_API_KEY"):
                if line.startswith(name + "="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    return None

# ★ 音声パス（未設定なら InputData/ の先頭を使う）。PII なのでコミット禁止。
AUDIO = "/Users/katohiroki/Documents/Labo/VScode/hakkutsu2026/speak-score/experiments/evaluation/InputData/TestInterview.mp4"
if not AUDIO:
    ind = EVAL_DIR / "InputData"
    cands = sorted([p for p in ind.glob("*") if p.suffix.lower() in
                    {".m4a", ".mp3", ".mp4", ".wav", ".mov", ".webm"}]) if ind.exists() else []
    AUDIO = str(cands[0]) if cands else ""
assert AUDIO and Path(AUDIO).exists(), "AUDIO に音声パスを設定してください（InputData/ に置くと自動検出）"

# 全候補で同じ WAV mono16k を使う
WAV = AUDIO
if Path(AUDIO).suffix.lower() != ".wav":
    WAV = str(Path(EVAL_DIR) / "_diar_tmp.wav")
    subprocess.run(["ffmpeg", "-y", "-i", AUDIO, "-ac", "1", "-ar", "16000", "-vn", WAV],
                   check=True, capture_output=True)

import soundfile as sf
audio, sr = sf.read(WAV, dtype="float32")
DUR = len(audio) / sr
print(f"AUDIO = {AUDIO}")
print(f"WAV   = {WAV}  ({DUR:.1f}s, sr={sr})")
print("OpenAI key :", "OK" if load_api_key() else "なし")
print("Gladia key :", "OK" if load_gladia_key() else "なし（A はスキップ）")


AUDIO = /Users/katohiroki/Documents/Labo/VScode/hakkutsu2026/speak-score/experiments/evaluation/InputData/TestInterview.mp4
WAV   = /Users/katohiroki/Documents/Labo/VScode/hakkutsu2026/speak-score/experiments/evaluation/_diar_tmp.wav  (218.1s, sr=16000)
OpenAI key : OK
Gladia key : OK


## 共通ヘルパ（アライメント・応募者特定・整形）

In [4]:
from collections import defaultdict

def turns_str(turns, limit=40):
    return "\n".join(f"  {s:6.2f}-{e:6.2f}  speaker_{spk}" for s, e, spk in turns[:limit])

def n_speakers(turns):
    return len(set(spk for _, _, spk in turns))

def assign_speakers(segments, turns):
    """Whisper セグメント(start,end,text) に話者を IoU 多数決で割当 → (s,e,text,spk)"""
    out = []
    for s, e, txt in segments:
        tally = defaultdict(float)
        for ts, te, spk in turns:
            ov = max(0.0, min(e, te) - max(s, ts))
            if ov > 0:
                tally[spk] += ov
        spk = max(tally, key=tally.get) if tally else None
        out.append((s, e, txt, spk))
    return out

def pick_applicant(turns):
    """最長累積発話の話者を応募者と仮定（ヒューリスティック）"""
    tot = defaultdict(float)
    for s, e, spk in turns:
        tot[spk] += e - s
    return max(tot, key=tot.get) if tot else None


## Whisper(verbatim) ベースライン — B/C はこれを話者割当して使う

In [ ]:
VERBATIM_PROMPT = "えーと、あのー、まあ、なんか、その、といったフィラーも省略せずそのまま書き起こす。"  # backend/src/services/transcription.py の _VERBATIM_PROMPT と完全一致（本番再現のため）
from openai import OpenAI
oai = OpenAI(api_key=load_api_key())

def whisper_verbatim(wav):
    r = oai.audio.transcriptions.create(
        model="whisper-1", file=open(wav, "rb"), language="ja", temperature=0,
        prompt=VERBATIM_PROMPT, response_format="verbose_json",
        timestamp_granularities=["segment"])
    segs = [(s.start, s.end, s.text) for s in (getattr(r, "segments", None) or [])]
    return r.text, segs

t0 = time.time()
whisper_text, whisper_segs = whisper_verbatim(WAV)
print(f"Whisper {time.time()-t0:.1f}s  fillers={len(find_fillers(whisper_text))}  segs={len(whisper_segs)}")
print(whisper_text)

## A. Gladia（STT 置換・話者分離組込み）

Gladia が **フィラーを保持するか**（Whisper verbatim と比較）も見る。保持しないなら #21/#22 の
disfluency 戦略には不利。

In [2]:
import httpx

def gladia_diarize(wav, key, languages=("ja",)):
    h = {"x-gladia-key": key}
    with open(wav, "rb") as f:
        up = httpx.post("https://api.gladia.io/v2/upload", headers=h,
                        files={"audio": (Path(wav).name, f, "audio/wav")}, timeout=180).json()
    body = {
        "audio_url": up["audio_url"],
        "model": "solaria-1",                       # 日本語対応（solaria-3 は欧州言語のみ）
        "language_config": {"languages": list(languages)},
        "diarization": True,
        "diarization_config": {"min_speakers": 1, "max_speakers": 3},
    }
    job = httpx.post("https://api.gladia.io/v2/pre-recorded", headers=h, json=body, timeout=60).json()
    result_url = job.get("result_url") or f"https://api.gladia.io/v2/pre-recorded/{job['id']}"
    for _ in range(150):
        r = httpx.get(result_url, headers=h, timeout=60).json()
        st = r.get("status")
        if st == "done":
            break
        if st == "error":
            raise RuntimeError(f"Gladia error: {r}")
        time.sleep(2)
    tr = r["result"]["transcription"]
    utts = [(u["start"], u["end"], u.get("speaker", 0), u["text"]) for u in tr.get("utterances", [])]
    return utts, tr.get("full_transcript", "")

gk = load_gladia_key()
if gk:
    t0 = time.time()
    g_utts, g_full = gladia_diarize(WAV, gk)
    g_turns = [(s, e, spk) for s, e, spk, _ in g_utts]
    print(f"Gladia {time.time()-t0:.1f}s  speakers={n_speakers(g_turns)}  "
          f"fillers={len(find_fillers(g_full))}  (Whisper verbatim比: {len(find_fillers(whisper_text))})")
    print("--- 話者区間 ---"); print(turns_str(g_turns))
    print("--- 応募者(最長話者)のテキスト ---")
    app = pick_applicant(g_turns)
    print("".join(t for _, _, spk, t in g_utts if spk == app))
else:
    print("GRADIA_API_KEY なし → A スキップ")
    g_turns = None


NameError: name 'load_gladia_key' is not defined

## B. sherpa-onnx（ローカルONNX・Whisper維持）

`num_clusters=-1`(自動) は**単一話者で過分割しやすい**（実測済み）。`threshold` を上げると分割が減る。
2話者音声では `num_clusters=2` 固定も試す。

In [5]:
import sherpa_onnx

def make_sd(num=-1, threshold=0.5, quant=True):
    seg = "model.int8.onnx" if quant else "model.onnx"
    cfg = sherpa_onnx.OfflineSpeakerDiarizationConfig(
        segmentation=sherpa_onnx.OfflineSpeakerSegmentationModelConfig(
            pyannote=sherpa_onnx.OfflineSpeakerSegmentationPyannoteModelConfig(
                model=str(MODELS_DIR / "sherpa-onnx-pyannote-segmentation-3-0" / seg))),
        embedding=sherpa_onnx.SpeakerEmbeddingExtractorConfig(
            model=str(MODELS_DIR / "3dspeaker_eres2net_base_16k.onnx")),
        clustering=sherpa_onnx.FastClusteringConfig(num_clusters=num, threshold=threshold),
        min_duration_on=0.3, min_duration_off=0.5)
    return sherpa_onnx.OfflineSpeakerDiarization(cfg)

def sherpa_diarize(num=-1, threshold=0.5):
    sd = make_sd(num, threshold)
    t0 = time.time()
    res = sd.process(audio).sort_by_start_time()
    dt = time.time() - t0
    return [(r.start, r.end, r.speaker) for r in res], dt

print("=== threshold スイープ（num=-1 自動）===")
for th in (0.5, 0.6, 0.7, 0.8):
    turns, dt = sherpa_diarize(num=-1, threshold=th)
    print(f"threshold={th}  speakers={n_speakers(turns)}  RTF={dt/DUR:.3f}  segs={len(turns)}")

print("\n=== 採用候補: num=-1, threshold=0.7 ===")
s_turns, s_dt = sherpa_diarize(num=-1, threshold=0.7)
print(turns_str(s_turns))


=== threshold スイープ（num=-1 自動）===
threshold=0.5  speakers=8  RTF=0.346  segs=33
threshold=0.6  speakers=6  RTF=0.338  segs=31
threshold=0.7  speakers=6  RTF=0.336  segs=31
threshold=0.8  speakers=6  RTF=0.339  segs=31

=== 採用候補: num=-1, threshold=0.7 ===
    0.42- 27.44  speaker_0
    1.03-  1.43  speaker_1
   28.38- 29.63  speaker_0
   31.08- 33.00  speaker_0
   33.56- 41.97  speaker_0
   42.76- 54.98  speaker_8
   65.25- 82.20  speaker_8
   82.70- 84.49  speaker_8
   85.11- 86.06  speaker_8
   86.62- 88.96  speaker_8
   96.02- 98.38  speaker_8
   99.46-120.97  speaker_8
  124.21-128.57  speaker_8
  129.51-130.15  speaker_10
  131.57-132.35  speaker_10
  133.21-135.98  speaker_11
  133.50-133.87  speaker_10
  136.72-138.68  speaker_8
  140.99-150.81  speaker_8
  148.95-150.07  speaker_0
  152.11-153.56  speaker_8
  154.39-178.80  speaker_8
  179.67-185.25  speaker_8
  185.88-186.57  speaker_8
  187.11-195.09  speaker_8
  196.91-208.44  speaker_8
  209.48-210.11  speaker_15
  210.11-211

## C. VADのみ ベースライン（Whisper維持・単一話者前提）

In [6]:
import librosa, numpy as np

def vad_turns(top_db=25):
    iv = librosa.effects.split(audio, top_db=top_db)
    return [(s / sr, e / sr, 0) for s, e in iv]   # すべて speaker 0 = 応募者

v_turns = vad_turns()
print(f"VADのみ  voiced区間={len(v_turns)}  （話者は常に1=応募者と仮定）")
print(turns_str(v_turns))


/Users/katohiroki/Documents/Labo/VScode/hakkutsu2026/speak-score/backend/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VADのみ  voiced区間=273  （話者は常に1=応募者と仮定）
    0.26-  0.58  speaker_0
    0.93-  1.89  speaker_0
    1.95-  3.84  speaker_0
    3.90-  4.70  speaker_0
    4.77-  4.96  speaker_0
    5.18-  8.06  speaker_0
    8.26-  9.82  speaker_0
    9.89- 10.08  speaker_0
   10.69- 11.42  speaker_0
   11.52- 11.71  speaker_0
   11.74- 12.99  speaker_0
   13.02- 13.25  speaker_0
   13.34- 13.66  speaker_0
   13.92- 15.68  speaker_0
   15.74- 15.94  speaker_0
   16.00- 16.54  speaker_0
   16.61- 17.09  speaker_0
   17.18- 17.86  speaker_0
   18.37- 18.66  speaker_0
   18.69- 18.85  speaker_0
   19.30- 19.68  speaker_0
   19.74- 22.98  speaker_0
   23.17- 24.35  speaker_0
   24.42- 26.62  speaker_0
   26.82- 27.23  speaker_0
   27.33- 27.52  speaker_0
   28.38- 28.96  speaker_0
   29.02- 29.57  speaker_0
   30.24- 30.53  speaker_0
   31.14- 33.06  speaker_0
   33.18- 33.54  speaker_0
   33.57- 34.59  speaker_0
   34.75- 34.94  speaker_0
   35.07- 35.55  speaker_0
   35.58- 36.16  speaker_0
   36.51- 37.70  s

## アライメント：Whisper セグメントに話者を割当（B/C 共通）

sherpa/VAD は Whisper を維持するので、話者区間を Whisper セグメントに IoU 多数決でマッピングして
`speaker` を付与 → 応募者セグメントだけを後段（disfluency/LLM）に渡す。

In [7]:
app_spk = pick_applicant(s_turns)
labeled = assign_speakers(whisper_segs, s_turns)
print(f"sherpa: 応募者=speaker_{app_spk}")
for s, e, txt, spk in labeled:
    mark = "★応募者" if spk == app_spk else f"speaker_{spk}"
    print(f"  [{s:6.2f}-{e:6.2f}] {mark}  {txt}")
print("\n--- 応募者セグメントのみ連結（後段へ渡すテキスト）---")
print("".join(txt for _, _, txt, spk in labeled if spk == app_spk))


sherpa: 応募者=speaker_8
  [  0.00- 10.68] speaker_0  はい。じゃあ、本題に入る前に少し注意点ということでですが、過去お話しいただいたところは、石川さんの話全部目を通してきてます。
  [ 10.68- 19.32] speaker_0  ですが、直接お伺いしたいなと思っているので、前に話してよってことも合わせて聞いてしまうと思うんですけど、はい、教えてください。
  [ 19.32- 28.36] speaker_0  質問の中で答えにくいところ、答えたくないところは無理して答えなくていいので、教えていただける範囲でお願いいたします。
  [ 28.36- 29.36] speaker_0  大丈夫ですか?
  [ 29.36- 30.36] speaker_0  はい、大丈夫です。
  [ 30.36- 36.36] speaker_0  はい。じゃあ、はじめのところは簡単なところからいきたいなと思うんですが、
  [ 36.36- 42.36] speaker_0  うちの会社の印象、どんな会社の印象に見えているかっていうところからお話し聞かせていただいていいですか?
  [ 42.36- 53.36] ★応募者  はい。会社の印象と、そうですね。御社が言われているように、事業家集団っていうのが言葉がぴったりで、私もそういうふうに感じていて、
  [ 54.36- 59.36] ★応募者  エンジニアとしてサーバーインターに参加させていただいたんですけども、
  [ 59.36- 75.36] ★応募者  結構他社さんとは異なっていて、機能の実装だとか、そういったところよりも前の部分、どうやって実装するのかだとか、それよりもエンジニアよりも前の、
  [ 75.36- 86.36] ★応募者  いわゆるPDMの範囲っていうんですかね。顧客がどういう課題を持っていて、それに対してどういう打ち手を打つのかだとか、それに対してPLを引くとか、
  [ 86.36- 94.36] ★応募者  そういったことをエンジニアのインターンでやるっていうのは、すごく自分とした踊りだというか、そうなんだと思ったので、
  [ 94.36-108.36] ★応募者  そこを実際に体験して、カイドルが高いビジネス職の方だったりとか、エンジ

## 比較サマリ（2026-06-21・石川の実面接2話者で評価）

| 候補 | 話者数 | 速度 | フィラー保持 | 単一/過分割 | 所見 |
|---|---|---|---|---|---|
| **A. Gladia** | 2（正） | 〜17s（API・最速） | 3（最多だが実態比は僅少） | 過分割なし | 話者数は当てるが**境界の帰属が甘く面接官が混入**。固有語STT誤り（サーマー/PTM）＋末尾「うん」幻聴（再現する・要後処理）。**新規外部依存＋課金**。 |
| **B. sherpa-onnx** | num=2:2 / auto:6 | 〜41-56s（CPU2スレ・ローカル） | 1（=Whisper） | autoは過分割だが**抽出は無害** | 境界(segmentation)が正確で**応募者の切り出しが最も綺麗**。GPU不要・ピーク約460MB・新キー不要。`num_clusters=2` で人数固定可。 |
| C. VADのみ | 1（分離不可） | 最速 | — | — | 2話者では面接官と混ざり使用不可。単一話者保証時のフォールバック止まり。 |

### 結論
- **候補は実質 A（Gladia全部）と B（sherpa num=2 + Whisper）の二択**。ハイブリッド（Gladia分離+Whisper）は両者の短所取りで除外。
- **評価の純度・運用の軽さ重視 → B**：応募者テキストに面接官混入なし、CPU/2GiB完結、外部依存・課金ゼロ。
- **filler信号＋速度重視で課金許容 → A**：ただし境界混入・固有語誤り・幻聴後処理が宿題。**キー/課金は石川(infra)判断**。
- **filler指標は現状ほぼ機能せず**（1〜3）。#21 disfluency の本丸は話者分離選択ではなく **STT強化＋検出器再設計**。

判断軸: 分離精度（境界の純度＞話者数の正しさ）/ フィラー保持(STT依存) / 速度・メモリ(CPU Cloud Run 2GiB) / 所有境界(新規外部依存・課金) / Eatas転用。


In [5]:
import re
from collections import Counter

# 末尾「うん」連発のような幻聴ループ（短い単位の6回以上連続）を検出・除去
REP = re.compile(r"(.{1,3}?)\1{5,}")
def is_halluc(t):
    m = REP.search(t)
    return bool(m) and len(m.group(0)) >= max(10, 0.5 * len(t))
def collapse_rep(t):
    return REP.sub(lambda m: m.group(1), t)
def fb(t):  # フィラー内訳（パターン別個数）
    return dict(Counter(h["text"] for h in find_fillers(t)))

def applicant_text(turns, segs):
    """話者区間 turns を Whisper セグメントに割当→最長話者(=応募者)のテキストを連結"""
    app = pick_applicant(turns)
    return "".join(txt for _, _, txt, spk in assign_speakers(segs, turns) if spk == app)

# A: Gladia STT そのもの（応募者発話のみ・幻聴除去） / B: Gladia 境界 + Whisper STT
if g_turns:
    g_app = pick_applicant(g_turns)
    A = "".join(("" if is_halluc(t) else collapse_rep(t)) for s, e, spk, t in g_utts if spk == g_app)
    B = applicant_text(g_turns, whisper_segs)
else:
    A = B = "(Gladia なし)"

# C: sherpa num=2（面接=2人を明示） / D: sherpa auto（cell-10 の s_turns 再利用）
c_turns, _ = sherpa_diarize(num=2, threshold=0.7)
C = applicant_text(c_turns, whisper_segs)
D = applicant_text(s_turns, whisper_segs)

print(f"Whisper全文(両話者): fillers={len(find_fillers(whisper_text))} 内訳={fb(whisper_text)}\n")
for label, txt, turns in [
    ("A. Gladia STT置換",        A, g_turns),
    ("B. Gladia分離+Whisper",    B, g_turns),
    ("C. sherpa(num=2)+Whisper", C, c_turns),
    ("D. sherpa(auto)+Whisper",  D, s_turns),
]:
    spk = n_speakers(turns) if turns else "-"
    print(f"--- {label}  [speakers={spk}] ---")
    print(f"  chars={len(txt)}  fillers={len(find_fillers(txt))}  内訳={fb(txt)}")
    print(f"  先頭: {txt[:80]}")
    print(f"  末尾: {txt[-80:]}\n")


NameError: name 'g_turns' is not defined

### 実行結果（2026-06-21・石川の実面接サンプル）

> ※上のセルを再実行すると同等の結果が再現する（Gladia/Whisperは都度API、sherpaはローカル）。下表は実測の記録。

| 方式 | speakers | 応募者chars | fillers(内訳) | 面接官の混入 | 固有語のSTT | 速度 |
|---|---|---|---|---|---|---|
| Whisper全文(基準) | — | — | 1 (`その`) | — | ◎ | 〜15s |
| **A. Gladia STT置換** | 2 | 923 | **3** (`あの`×2,`その`) | ⚠️ 冒頭に混入 | △ `サーマー`/`PTM` | 〜17s |
| **B. Gladia分離+Whisper** | 2 | 984 | 1 (`その`) | ⚠️ 末尾に混入 | ◎ | 〜32s |
| **C. sherpa(num=2)+Whisper** | 2 | 915 | 1 (`その`) | ✅ なし | ◎ | 〜56s |
| D. sherpa(auto)+Whisper | 6 | 915 | 1 (`その`) | ✅ なし | ◎ | 〜56s |

**所見**
1. **応募者の切り出しは C（sherpa num=2）が最良**。A は冒頭、B は末尾に面接官発話が混入する（Gladia は話者数2は当てるが境界の帰属が甘い）。
2. **C と D は応募者テキストが完全一致** → sherpa の過分割（6話者）は抽出品質に無害。境界(segmentation)は正確で、過分割は clustering 段の問題に過ぎない。`num_clusters=2` は人数既知時の保険。
3. **ハイブリッド B は両者の悪いとこ取り**（Gladiaのキー/課金を払いつつ境界混入を引き継ぎ、filler は Whisper の1に戻る）→ 候補から除外。
4. **filler 指標は現状ほぼ機能していない**。3.5分の自発発話で 1〜3 は実態と乖離。`whisper-1` が言い淀みを正規化で除去＋検出器が9パターン部分一致のみ（伸ばし/言い直し/`うん`/`えっと` 非対応）。**#21 の本丸は STT 強化＋検出器再設計**で、話者分離の選択では filler は動かない。

→ 実質 **A（Gladia全部・filler+速度）vs C（sherpa・純度+所有境界の軽さ）** の二択。キー/課金は石川(infra)判断。


## 文字起こし × 話者クラスタリングの突き合わせ（目視確認用）

2パイプラインの全発話を**話者ラベル付き**で並べ、クラスタリングの当たり外れを目で見る。

- **A. Gladia**: STT と話者分離が一体（Gladia 自身のテキスト・話者）。
- **C. sherpa(num=2) + Whisper**: Whisper のセグメントに sherpa の話者を IoU 割当。

`★応募者` = 最長累積発話の話者（`pick_applicant`）。混入＝面接官の発話が `★応募者` 側に入っていないかを見る。


In [ ]:
# 前提: cell-2(setup) / cell-4(helpers) / cell-6(Whisper) を先に実行しておくこと。
#       迷ったらメニューの Kernel → Restart & Run All で全セルを上から流すのが確実。
if "whisper_segs" not in globals():
    raise RuntimeError("先に cell-6（Whisper ベースライン）を実行してください（whisper_segs が未定義）。")

# sherpa の話者区間が無ければここで作る（cell-10 の sherpa_diarize が必要）
if "c_turns" not in globals():
    c_turns, _ = sherpa_diarize(num=2, threshold=0.7)

def show_labeled(title, items, applicant):
    """items: list[(start, end, speaker, text)] を話者ラベル付きで出力"""
    print(f"\n========== {title}  (応募者=speaker_{applicant}) ==========")
    for s, e, spk, txt in items:
        mark = "★応募者 " if spk == applicant else f"　面接官?(s{spk})"
        print(f"[{s:6.1f}-{e:6.1f}] {mark} {txt}")

# --- C. sherpa(num=2) + Whisper（Whisperテキストに話者割当）---
c_app = pick_applicant(c_turns)
items_c = [(s, e, spk, txt) for s, e, txt, spk in assign_speakers(whisper_segs, c_turns)]
show_labeled("C. sherpa(num=2) + Whisper STT", items_c, c_app)

# --- A. Gladia（cell-8 を実行済みなら表示。未実行ならスキップ）---
if globals().get("g_turns"):
    g_app = pick_applicant(g_turns)
    show_labeled("A. Gladia（STT＋話者分離 一体）", g_utts, g_app)
else:
    print("\n（A. Gladia はスキップ: 表示するには cell-8 を実行してください）")


## フィラー検出は Whisper の prompt 次第で激変する（1 → 33）

`find_fillers` の個数は STT エンジンや話者分離ではなく、**Whisper の `prompt` 内容**でほぼ決まる。
同じ音声・同じ検出器でも、プロンプトを変えるだけで 1 → 33 に変わる（下セルで再現）。

→ **#21 disfluency の本丸は「Whisperのプロンプト設計（＋検出器）」**。話者分離の選択では filler は動かない。
`whisper-1` + フィラー語プロンプトなら Gladia(3個)を大きく上回るので、**filler を理由に Gladia を選ぶ必要はない**。
（注: フィラー語羅列プロンプトは過剰プライミングで実発話以上に拾う可能性があり、保持率の妥当性は #21 で要検証。）


In [ ]:
# 同一音声・同一検出器で prompt だけ変えてフィラー個数を比較（whisper-1 を3回呼ぶ）
PROMPTS = {
    "prompt無し(main/transcribe.py既定)": None,
    "verbatim指示文(本番transcription.pyと完全一致)": "えーと、あのー、まあ、なんか、その、といったフィラーも省略せずそのまま書き起こす。",
    "フィラー語の羅列のみ":                 "えー、えーと、あのー、あの、その、まあ、なんか、ええと。",
}
for label, p in PROMPTS.items():
    kw = dict(model="whisper-1", file=open(WAV, "rb"), language="ja", temperature=0,
              response_format="verbose_json", timestamp_granularities=["segment"])
    if p:
        kw["prompt"] = p
    txt = oai.audio.transcriptions.create(**kw).text
    print(f"{label:30s} fillers={len(find_fillers(txt)):2d}  内訳={fb(txt)}  chars={len(txt)}")

# 実測(2026-06-21・旧プロンプト版): prompt無し=1 / verbatim指示文(旧)=1 / フィラー語羅列=33({'あの':30,'その':3})
# 本番プロンプトに揃えた版は再実行して更新すること（本番再現実験: docs/ai_log/2026-06-21_prod_vs_local_whisper_prompt_mismatch.md 参照、fillers=28で再現）

## 3者の文字起こし全文を確認【自己完結セル】

下のコードは **0.セットアップ（cell-2相当）だけ実行済みなら単体で動く**（他セル不要）。

- ①whisper-1 prompt無し … `transcribe.py` 既定＝main 相当（句読点なし・ゴミ語混入しやすい）
- ②whisper-1 verbatimプロンプト … このNB現状（句読点あり・最もクリーン）
- ③Gladia STT … Gladia 自身の全文（固有名詞誤り＋末尾「うん」幻聴あり）

実測(2026-06-21): fillers ①1 / ②1 / ③3、いずれもフィラーはほぼ残らない。


In [8]:
# 【自己完結】前提は 0.セットアップ（WAV / load_api_key / load_gladia_key / find_fillers）のみ。
from openai import OpenAI as _OpenAI
import httpx as _httpx, time as _time
_oai = _OpenAI(api_key=load_api_key())

def _whisper(prompt=None):
    kw = dict(model="whisper-1", file=open(WAV, "rb"), language="ja", temperature=0,
              response_format="verbose_json", timestamp_granularities=["segment"])
    if prompt:
        kw["prompt"] = prompt
    return _oai.audio.transcriptions.create(**kw).text

def _gladia_full():
    key = load_gladia_key()
    if not key:
        return "(GRADIA_API_KEY なし → スキップ)"
    h = {"x-gladia-key": key}
    with open(WAV, "rb") as f:
        up = _httpx.post("https://api.gladia.io/v2/upload", headers=h,
                         files={"audio": (Path(WAV).name, f, "audio/wav")}, timeout=180).json()
    body = {"audio_url": up["audio_url"], "model": "solaria-1",
            "language_config": {"languages": ["ja"]}, "diarization": True,
            "diarization_config": {"min_speakers": 1, "max_speakers": 3}}
    job = _httpx.post("https://api.gladia.io/v2/pre-recorded", headers=h, json=body, timeout=60).json()
    url = job.get("result_url") or f"https://api.gladia.io/v2/pre-recorded/{job['id']}"
    for _ in range(150):
        r = _httpx.get(url, headers=h, timeout=60).json()
        if r.get("status") == "done":
            break
        if r.get("status") == "error":
            raise RuntimeError(r)
        _time.sleep(2)
    return r["result"]["transcription"].get("full_transcript", "")

def _show(title, txt):
    print("=" * 78)
    print(f"{title}   fillers={len(find_fillers(txt))}  chars={len(txt)}")
    print("=" * 78)
    print(txt, "\n")

_show("① whisper-1 prompt無し（main/transcribe.py 相当）", _whisper(None))
_show("② whisper-1 verbatimプロンプト（本番transcription.pyと完全一致）",
      _whisper("えーと、あのー、まあ、なんか、その、といったフィラーも省略せずそのまま書き起こす。"))
_show("③ Gladia STT 全文", _gladia_full())

① whisper-1 prompt無し（main/transcribe.py 相当）   fillers=1  chars=1313
本題に入る前に少し注意点という ことですが 過去お話しいただいた ところは石川さんの話全部目を 通してきてます ですが 直接お伺い したいなと思っているので 前に 話してよってことも併せて聞いて しまうと思うんですけど 教えて ください 質問の中で答えにくい ところ 答えたくないところは無理 して答えなくていいので 教えて いただける範囲でお願いいたします 大丈夫そうですか はい 大丈夫です はい じゃあ 初めのところは簡単な ところからいきたいなと思うん ですが うちの会社の印象 どんな 会社の印象に見えているかという ところからお話し聞かせていただ いていいですか はい 会社の印象 そうですね 御 者が言われているように事業家集団 というのが言葉がぴったりで 私も そういうふうに感じていて エンジニア としてサーバーインターに参加 させていただいたんですけども 結構他社さんとは異なっていて 機能の実装だとか そういったところ よりも前の部分 どうやって実装 するのかだとか それよりもエンジニア よりも前のいわゆるPDMの範囲 というんですかね 顧客がどういう 課題を持っていて それに対して どういう打ち手を打つのかだとか それに対してPLを引くとか そう いったことをエンジニアのインター でやるっていうのはすごく 自分 とした踊りだというか そうな なと思ったことで そこを実際に 体験して カイドルが高いビジネス 職の方だったりとか エンジニア なのにそういったことの話まで されるようなエンジニアの方っていう のを見て そういったことを会社 として上げていて そこが強みなんだ なっていうふうに感じるとともに 私もそういったふうな人間になり たいと思ってるので こういう環境 にいると こういう環境が一番自分 でも成長できるんだろうなっていう ふうに感じたので 一番事業家集団 っていうのがすごく印象に思って ます おだしょー 他はどうですか もしあれば 事業家の印象以外は 三沢 そうですね そういった部分 が強いのかなっていうふうに感じ てたんですけど エンジニアの方 にお話を伺って 技術的な部分も しっかりとしてるっていうのが 聞

## フィラー検出: パターンマッチ vs 形態素解析（精度比較）

現状の `find_fillers`（backend `transcription.py` / このNBの `transcribe.py`）は9パターンの部分一致のみ。
[ADR 004](../adr/004_verbatim_transcription_and_disfluency.md) で「精緻化は形態素 + UniDic『感動詞-フィラー』」と
予告されていた方式を実際に試し、パターンマッチと比較する。

**API呼び出しは行わない**: 本番再現実験（[docs/ai_log/2026-06-21_prod_vs_local_whisper_prompt_mismatch.md](../ai_log/2026-06-21_prod_vs_local_whisper_prompt_mismatch.md)）
で取得済みの実本番テキスト（石川面接サンプル・backendの`_VERBATIM_PROMPT`と完全一致のプロンプトで3回再現確認済み）を
そのまま `PROD_TEXT` として埋め込んで使う。Whisperを再実行すると時間がかかるため。

比較する3手法:
- **A. パターンマッチ（既存）**: `FILLER_PATTERNS` の部分一致。
- **B. 形態素解析のみ**: fugashi(MeCab)+unidic-lite で `pos1=感動詞, pos2=フィラー` のトークンだけを採用。
- **C. ハイブリッド**: B に加えて、連体詞「あの/その」のうち直後が読点・補助記号・文末のものをフィラーとみなす
  （直後が名詞などで「あの人」のような真の連体詞用法なら除外）。

事前準備: `uv pip install --python .venv/bin/python fugashi unidic-lite`（backendの.venvに追加済み・軽量・モデル同梱でDL不要）。

In [9]:
import fugashi
from collections import Counter
from transcribe import FILLER_PATTERNS  # 0.セットアップで EVAL_DIR が sys.path に入っている前提

tagger = fugashi.Tagger()  # uv pip install --python .venv/bin/python fugashi unidic-lite

# 本番再現実験で取得済みの実テキスト（2026-06-21・石川面接サンプル・backend _VERBATIM_PROMPT と完全一致設定で3回連続同一出力）。
# Whisper再呼び出しはしない。PII のためこのテキストはサンプル音声と同様、リポジトリ外では取り扱い注意。
PROD_TEXT = (
    "はい。じゃあ、そうだ、本題に入る前に少し注意点ということで、 すが、過去お話しいただいたところは、"
    "石川さんの話全部目を通してきてます。 ですが、直接お伺いしたいなと思っているので、前に話してよってことも"
    "合わせて聞いてしまうと思うんですけど、 はい、教えてください。質問の中で答えにくいところ、答えたくないところは"
    "無理して答えなくていいので、 教えていただける範囲でお願いいたします。 大丈夫ですか? はい、大丈夫です。"
    " はい。じゃあ、はじめのところは、簡単なところからいきたいなと思うんですが、"
    " うちの会社の印象、どんな会社の印象に見えているかっていうところからお話し聞かせていただいていいですか?"
    " はい、会社の印象と、そうですね、御社が言われているように、"
    " あの、事業家集団っていうのが言葉がぴったりで、私もそういう風に感じていて、"
    " あの、エンジニアとしてサーバーインターに参加させていただいたんですけども、"
    " 結構他社さんとは異なっていて、 あの、機能の実装だとか、そういったところよりも前の部分、"
    " あの、どうやって実装するのかだとか、それよりもエンジニアよりも前の、"
    " あの、いわゆるPDMの範囲っていうんですかね。 あの、顧客がどういう課題を持っていて、"
    " あの、それに対してどういう打ち手を打つのかだとか、それに対してPLを引くとか、"
    " そういったことをこう、エンジニアのインターンでやるっていうのはすごく、"
    " あの、自分とした踊りだというか、あ、そうなのかと思ったので、 そこを実際に体験して、"
    " あの、カイドラが高い、あの、ビジネス職の方だったりとか、"
    " あの、エンジニアなどにそういったことの話までこう、 あの、されるようなエンジニアの方っていうのを見て、"
    " やっぱりそういったことを会社として、あの、やって、上げていて、 そこが強みなんだなっていう風に感じるとともに、"
    " 私も、あの、そういった風な人間になりたいと思ってるので、"
    " あの、こういう環境にいると、あの、こういう環境が一番自分でも成長できるんだろうなっていう風に感じたので、"
    " 一番そう、事業家集団っていう部分がすごく、あの、印象に思っています。 うーん。他はどうですか?"
    " 他? もしあれ、もしあれば、事業家の印象以外は。 あ、そうですね。そういった部分が強いのかなっていう風に感じてたんですけど、"
    " うん。 あの、ちなみにエンジニアの方にお話を伺って、 技術的な部分も、あの、しっかりとしてるっていうのが聞いていて、"
    " へー。 あの、他の企業さんを、あの、先行進ませていただいてるんですけども、"
    " その中でも結構その、えっと、技術的な部分に関しては、 あの、なんて言うんですかね。 放置じゃないですけど、"
    " あの、道は渡すけども、ある程度自分で進めてねっていう部分が結構大きくて、"
    " もちろんそれも、もちろん自分でやるのは前提なんですけども、"
    " あの、エンジニアリングの基礎の部分っていうのはやっぱり、 あの、それではどうしようもできない部分、"
    " 雑学的な部分が必要だなっていう風に感じていて、 私もエンジニア、えっと、大学で学んできたっていう分ではないので、"
    " そういったところから抜け落ちてるなって実感があるので、 そういったところからしっかりやって、"
    " あの、技術、最初の何年かを叩き込まれたっていうのを、 2、3人言われてたので、"
    " そこは、あの、相反するところかなと思いつつも、 どっちも大事にされてるんだなっていう印象ではあります。"
    " うーん。 技術、事業とですね。 うんうん。 技術ってことで、はい。 分かりました。ありがとうございます。"
)


def find_fillers_pattern(text):
    """A. 既存パターンマッチ（FILLER_PATTERNS の部分一致・最長一致非重複）"""
    hits = []
    i, n = 0, len(text)
    while i < n:
        matched = next((p for p in FILLER_PATTERNS if text.startswith(p, i)), None)
        if matched:
            hits.append((matched, i, i + len(matched)))
            i += len(matched)
        else:
            i += 1
    return hits


def tokenize_with_spans(text):
    """fugashi は空白を読み飛ばすため、text.find で実際の char offset を復元する。"""
    out, cursor = [], 0
    for w in tagger(text):
        s = text.find(w.surface, cursor)
        e = s + len(w.surface)
        out.append((w, s, e))
        cursor = e
    return out


def find_fillers_morph(text):
    """B. 形態素解析のみ（POS=感動詞-フィラー）"""
    return [(w.surface, s, e) for w, s, e in tokenize_with_spans(text)
            if w.feature.pos1 == "感動詞" and w.feature.pos2 == "フィラー"]


def find_fillers_hybrid(text):
    """C. B + 連体詞「あの/その」の文脈判定（直後が読点/補助記号/文末ならフィラー、名詞が続けば除外）"""
    toks = tokenize_with_spans(text)
    hits = []
    for i, (w, s, e) in enumerate(toks):
        f = w.feature
        if f.pos1 == "感動詞" and f.pos2 == "フィラー":
            hits.append((w.surface, s, e))
        elif f.pos1 == "連体詞" and w.surface in ("あの", "その"):
            nxt = toks[i + 1][0] if i + 1 < len(toks) else None
            if nxt is None or nxt.feature.pos1 == "補助記号":
                hits.append((w.surface, s, e))
    return hits


pat_hits = find_fillers_pattern(PROD_TEXT)
morph_hits = find_fillers_morph(PROD_TEXT)
hybrid_hits = find_fillers_hybrid(PROD_TEXT)


def report(name, hits):
    print(f"{name}: {len(hits)}件  内訳={dict(Counter(h[0] for h in hits))}")


report("A. パターンマッチ(既存)", pat_hits)
report("B. 形態素解析のみ(POS=感動詞/フィラー)", morph_hits)
report("C. ハイブリッド(フィラーPOS + あの/その文脈判定)", hybrid_hits)

pat_spans = {(s, e) for _, s, e in pat_hits}
hybrid_spans = {(s, e) for _, s, e in hybrid_hits}
print()
print("Cのみ検出(Aには無い):", [PROD_TEXT[s:e] for s, e in sorted(hybrid_spans - pat_spans)])
print("Aのみ検出(Cが除外):  ", [PROD_TEXT[s:e] for s, e in sorted(pat_spans - hybrid_spans)])
for s, e in sorted(pat_spans - hybrid_spans):
    print(f"  除外された文脈: …{PROD_TEXT[max(0,s-10):e+10]}…")

# 既知の取りこぼし: 「えっと」はどちらの手法でも正しく拾えない
print()
print("--- 「えっと」の実タグ（A/B/C 共通の弱点） ---")
for w, s, e in tokenize_with_spans(PROD_TEXT):
    if "えっ" in w.surface:
        print(f"  {w.surface!r}  pos={w.feature.pos1}/{w.feature.pos2}  パターン一致={'えっと' in FILLER_PATTERNS}")

A. パターンマッチ(既存): 29件  内訳={'あの': 27, 'その': 2}
B. 形態素解析のみ(POS=感動詞/フィラー): 0件  内訳={}
C. ハイブリッド(フィラーPOS + あの/その文脈判定): 28件  内訳={'あの': 27, 'その': 1}

Cのみ検出(Aには無い): []
Aのみ検出(Cが除外):   ['その']
  除外された文脈: …てるんですけども、 その中でも結構その、えっ…

--- 「えっと」の実タグ（A/B/C 共通の弱点） ---
  'えっ'  pos=感動詞/一般  パターン一致=False
  'えっ'  pos=感動詞/一般  パターン一致=False


### 結果（2026-06-21・石川面接サンプル、実測）

| 手法 | 件数 | 内訳 |
|---|---|---|
| A. パターンマッチ（既存） | 29 | あの×27, その×2 |
| B. 形態素解析のみ（POS=感動詞-フィラー） | **0** | — |
| C. ハイブリッド（B + あの/その文脈判定） | 28 | あの×27, その×1 |

**B（形態素解析のみ）は単独では使えない。** unidic-liteは短い「あの/その」を曖昧性解消で
ほぼ全て連体詞（真の指示詞用法）と判定し、感動詞-フィラーのタグが付くのは「あのー」のように
伸ばした場合に限られる。この実音声では全29件の「あの/その」が連体詞判定 → 純粋なPOSフィルタは再現率0%。

**C（ハイブリッド）が最も精度が高い。** 感動詞-フィラーPOS に加え、連体詞「あの/その」のうち直後が
読点・文末のものをフィラー扱いすることで、Aと同等の再現率（28/29）を保ったまま、Aが誤検出していた
「その中でも結構その、えっと…」の **「その」（真の連体詞＝指示詞用法、直後に名詞「中」）を正しく除外**できた。
パターンマッチには文脈判断が無いため、この種の誤検出は原理的に直せない。

**残課題（A/B/C共通の弱点）**: 「えっと」はfugashi+unidic-liteが「えっ」(感動詞-一般)+「と」(助詞)に
誤分割し、`FILLER_PATTERNS` にも未登録のため**3手法とも検出できない**。本番テキストに2箇所登場している
（実害あり）。対処は (1) `FILLER_PATTERNS` に "えっと" を追加（即効性あり）、(2) ユーザー辞書で
「えっと」を1トークンに強制する、のいずれか。

**結論**: 形態素解析は「あの/その」のような曖昧語の**誤検出を減らす**（精度＝precision向上）には有効だが、
それ単独で導入すると感動詞-フィラーPOSの再現率が低すぎて使えない。**パターンマッチを土台に、
連体詞の文脈フィルタ（C方式）を上乗せするハイブリッドが現実的な改善案**。
「えっと」の語彙追加は手法に関係なく今すぐ直せる別の改善点。